In [20]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

True

In [21]:
client = Anthropic()
model = "claude-haiku-4-5"

In [22]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

# ------------------------------- Evaluation Logic -------------------------------

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
    """
    messsages = []
    add_user_message(messsages, prompt)
    return chat(messsages)

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [23]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
        """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [24]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

### Running Evaluation

In [25]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [26]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Region Extractor from S3 URL\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_aws_region(s3_url: str) -> Optional[str]:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - Virtual-hosted-style: s3://my-bucket.s3.us-west-2.amazonaws.com/key\n    - Path-style: s3://s3.us-west-2.amazonaws.com/my-bucket/key\n    - S3 URI: s3://bucket-name/key (region cannot be determined)\n    \n    Args:\n        s3_url (str): The S3 bucket URL\n        \n    Returns:\n        Optional[str]: The AWS region (e.g., 'us-west-2') or None if not found\n        \n    Examples:\n        >>> extract_aws_region('s3://my-bucket.s3.us-west-2.amazonaws.com/key')\n        'us-west-2'\n        >>> extract_aws_region('s3://s3.us-west-2.amazonaws.com/my-bucket/key')\n        'us-west-2'\n        >>> extract_aws_region('s3://my-bucket/

## Implementing a Model Grader

In [27]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Integrating Grading into Your Workflow

In [28]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [29]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [30]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


Average score: 7


In [31]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_s3_region(url: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Args:\n        url: S3 bucket URL in formats like:\n             - s3://my-bucket.s3.us-west-2.amazonaws.com/key\n             - https://my-bucket.s3.us-west-2.amazonaws.com/key\n             - https://s3.us-west-2.amazonaws.com/my-bucket/key\n             - s3://my-bucket/key (uses default region us-east-1)\n    \n    Returns:\n        AWS region string (e.g., 'us-west-2')\n    \n    Raises:\n        ValueError: If region cannot be extracted from the URL\n    \"\"\"\n    \n    # Pattern 1: Virtual-hosted-style URLs (s3://bucket.s3.region.amazonaws.com/key)\n    pattern1 = r'\\.s3[.-]([a-z0-9\\-]+)\\.amazonaws\\.com'\n    match = re.search(pattern1, url)\n    if match:\n        return match.g